# Modele de Propension a l'Achat – AnyCompany Food & Beverage

**Objectif** : Predire la probabilite qu'une transaction soit un achat (Sale) vs un autre type (Refund, Investment, etc.)

**Modele** : RandomForestClassifier

**Donnees** : `ANYCOMPANY_LAB.ANALYTICS.VENTES_ENRICHIES`

In [ ]:
%%sql -r df_ventes
SELECT
    TRANSACTION_ID,
    TRANSACTION_TYPE,
    AMOUNT,
    PAYMENT_METHOD,
    REGION,
    ANNEE,
    TRIMESTRE,
    MOIS,
    JOUR_SEMAINE,
    PENDANT_PROMO,
    COALESCE(DISCOUNT_PERCENTAGE, 0) AS DISCOUNT_PERCENTAGE,
    COALESCE(CAMPAIGN_BUDGET, 0) AS CAMPAIGN_BUDGET,
    COALESCE(CAMPAIGN_REACH, 0) AS CAMPAIGN_REACH,
    COALESCE(CAMPAIGN_CONVERSION_RATE, 0) AS CAMPAIGN_CONVERSION_RATE,
    CASE WHEN TRANSACTION_TYPE = 'Sale' THEN 1 ELSE 0 END AS IS_SALE
FROM ANYCOMPANY_LAB.ANALYTICS.VENTES_ENRICHIES

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

print(f"Shape: {df_ventes.shape}")
print(f"\nDistribution de la cible:")
print(df_ventes['IS_SALE'].value_counts())
print(f"\nTaux d'achat: {df_ventes['IS_SALE'].mean():.2%}")

In [ ]:
le_payment = LabelEncoder()
le_region = LabelEncoder()

df = df_ventes.copy()
df['PAYMENT_ENC'] = le_payment.fit_transform(df['PAYMENT_METHOD'].astype(str))
df['REGION_ENC'] = le_region.fit_transform(df['REGION'].astype(str))
df['PENDANT_PROMO_INT'] = df['PENDANT_PROMO'].astype(int)

feature_cols = ['AMOUNT', 'PAYMENT_ENC', 'REGION_ENC', 'ANNEE', 'TRIMESTRE',
                'MOIS', 'JOUR_SEMAINE', 'PENDANT_PROMO_INT',
                'DISCOUNT_PERCENTAGE', 'CAMPAIGN_BUDGET', 'CAMPAIGN_REACH',
                'CAMPAIGN_CONVERSION_RATE']

X = df[feature_cols].fillna(0)
y = df['IS_SALE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Distribution train: {y_train.value_counts().to_dict()}")
print(f"Distribution test: {y_test.value_counts().to_dict()}")

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("=" * 50)
print("RAPPORT DE CLASSIFICATION")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Non-Sale', 'Sale']))

auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC Score: {auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
axes[0].set_xlabel('Predit')
axes[0].set_ylabel('Reel')
axes[0].set_title('Matrice de Confusion')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Non-Sale', 'Sale'])
axes[0].set_yticklabels(['Non-Sale', 'Sale'])

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Importance des Features')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("INTERPRETATION BUSINESS")
print("=" * 60)

importances_sorted = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 5 facteurs influencant l'achat:")
for i, (feat, imp) in enumerate(importances_sorted.head(5).items()):
    print(f"  {i+1}. {feat}: {imp:.4f}")

print(f"\nPerformance du modele:")
print(f"  AUC: {auc:.4f}")
print(f"  Le modele identifie correctement les acheteurs potentiels")

print(f"\nRecommandations:")
print(f"  - Cibler les clients avec une forte probabilite d'achat")
print(f"  - Optimiser les campagnes sur les facteurs cles identifies")
print(f"  - Personnaliser les offres selon le profil de propension")

In [ ]:
%%sql -r ctx
USE DATABASE ANYCOMPANY_LAB;
USE SCHEMA ANALYTICS;